# Configurations

In [1]:
BOOTSTRAP_SERVER = "10.67.22.102:19092" # Kafka port 19092 is dedicated to VMs in CloudVeneto

TOPIC_STREAM = "topic_stream"
TOPIC_RESULTS = "topic_results"

SCHEDULER_IP = "10.67.22.102"
WORKER_IPS = ["10.67.22.58", "10.67.22.244"]
N_WORKERS = len(WORKER_IPS)

WORKER_PUBLISH_INTERVAL_SEC = 5.0

# Topics setup

KafkaAdminClient options: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaAdminClient.html

Topic configs: https://kafka.apache.org/42/configuration/topic-configs/


In [2]:
from kafka.admin import KafkaAdminClient
# from kafka.errors import UnknownTopicOrPartitionError

print("Creating topics...")
kafka_admin = KafkaAdminClient(
    bootstrap_servers=BOOTSTRAP_SERVER,
    client_id = "kafka_admin_client",
    security_protocol = "PLAINTEXT"
)

try:
    kafka_admin.delete_topics([TOPIC_STREAM, TOPIC_RESULTS])
except UnknownTopicOrPartitionError:
    pass
kafka_admin.create_topics({
    TOPIC_STREAM: {
        "num_partitions": N_WORKERS,
        "replication_factor": 1,
    },
    TOPIC_RESULTS: {
        "num_partitions": 1,
        "replication_factor": 1,
    }
})
print("Waiting for topics...")
kafka_admin.wait_for_topics([TOPIC_STREAM, TOPIC_RESULTS]) # Block until each of the given topics is ready to use.
print("Topics ready")
print("List of topics:", kafka_admin.list_topics())
kafka_admin.close() # Close the KafkaAdminClient connection to the Kafka broker

Creating topics...
Waiting for topics...
Topics ready
List of topics: ['topic_results', 'topic_stream', '__consumer_offsets']


# Dask cluster setup

Dask needs to be installed in all nodes

Documentation: https://docs.dask.org/en/stable/deploying-ssh.html and https://docs.dask.org/en/stable/_modules/distributed/client.html

In [3]:
from dask.distributed import Client, SSHCluster

print("Creating Dask SSH cluster...")
dask_cluster = SSHCluster(
    [SCHEDULER_IP] + WORKER_IPS,
    connect_options = {"known_hosts": None}, # disable server host key validation
    worker_options = {"n_workers": 1, "nthreads": 1}, # Keywords to pass on to workers
    scheduler_options={"port": 8786, "dashboard_address": "0.0.0.0:8797"} # Keywords to pass on to scheduler
)

dask_client = Client(dask_cluster, name="dask_client")
dask_client

Creating Dask SSH cluster...


2026-07-12 20:44:41,139 - distributed.deploy.ssh - INFO - 2026-07-12 20:44:41,138 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-07-12 20:44:41,160 - distributed.deploy.ssh - INFO - 2026-07-12 20:44:41,160 - distributed.scheduler - INFO - State start
2026-07-12 20:44:41,164 - distributed.deploy.ssh - INFO - 2026-07-12 20:44:41,163 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.67.22.102:8786
2026-07-12 20:44:41,660 - distributed.deploy.ssh - INFO - 2026-07-12 20:44:41,636 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.244:34633'
2026-07-12 20:44:41,746 - distributed.deploy.ssh - INFO - 2026-07-12 20:44:41,711 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.58:42727'
2026-07-12 20:44:41,967 - distributed.deploy.ssh - INFO - 2026-07-12 20:44:41,942 - distributed.diskutils - INFO - Found stale lock file and directory '/t

<Client: 'tcp://10.67.22.102:8786' processes=1 threads=1, memory=1.92 GiB>

# Workers code

In [4]:
def process_batch(bytestring):
    import numpy as np
    
    SAMPLES_PER_SCAN = 2048 # Number of complex samples in each scan
    BYTES_PER_SCAN = 8 * SAMPLES_PER_SCAN
    SAMPLING_FREQ_HZ = 2e6 # ADC readout frequency

    batch_size = len(bytestring)
    assert batch_size % BYTES_PER_SCAN == 0
    n_scans = batch_size // BYTES_PER_SCAN

    # Split raw bytes into I and Q float32 arrays
    i_data = np.frombuffer(bytestring[:batch_size//2], dtype="<f4") # < = little-endian, f4 = float32
    q_data = np.frombuffer(bytestring[batch_size//2:], dtype="<f4")
    complex_signal = np.empty(i_data.shape, dtype=np.complex64) # this way avoids intermediate allocation
    complex_signal.real = i_data
    complex_signal.imag = q_data

    # Shape the signal into a matrix to perform vectorized FFT row-wise
    matrix = complex_signal.reshape((n_scans, SAMPLES_PER_SCAN))
    # Compute centered frequency spectrum
    spectra = np.fft.fftshift(np.fft.fft(matrix, axis=1), axes=1)
    power_spectra = np.abs(spectra) ** 2

    # Compute mean and second moment of power across scans
    power_means = np.mean(power_spectra, axis=0)
    power_M2s = np.sum((power_spectra - power_means) ** 2, axis=0)

    # Compute frequency bins
    frequencies = np.fft.fftshift(np.fft.fftfreq(SAMPLES_PER_SCAN, d = 1 / SAMPLING_FREQ_HZ)) # could be optimized out of the function

    return {
        "n_scans": n_scans,
        "frequencies": frequencies,
        "means": power_means,
        "M2s": power_M2s
    }

def worker_function(worker_id, bootstrap_servers, input_topic, output_topic, publish_interval_sec):
    # Worker imports are inside the function so they execute on the worker process
    import json
    import numpy as np
    import time
    from kafka import KafkaConsumer, KafkaProducer

    # Setup consumer. This will read data from the DAQ
    consumer = KafkaConsumer(
        input_topic,
        bootstrap_servers=bootstrap_servers,
        client_id = f"worker{worker_id}_consumer", # Appears in server-side logs
        group_id = 'dask_processing_group',        # consumer group to join for dynamic partition assignment
        group_instance_id = f"worker{worker_id}_consumer_instance", # Make this consumer a static member of the group (not really necessary)
        value_deserializer = None,                 # Deserialization will be performed manually
        max_partition_fetch_bytes = 32*1024*1024,  # This size must be at least as large as the maximum message size the server allows
        auto_offset_reset='earliest',              # Bring the reading offset to the earliest message
        security_protocol = "PLAINTEXT", 
    )
    
    # Setup producer. This will publish data for the dashboard
    producer = KafkaProducer(
        bootstrap_servers=bootstrap_servers,
        client_id = f"worker{worker_id}_producer", # Appears in server-side logs
        value_serializer=lambda v: json.dumps(v).encode('utf-8'),
        enable_idempotence = True,                 # Ensure that exactly one copy of each message is written in the stream
        compression_type=None,                     # No compression
        batch_size = 0,                            # Disable batching, we will do our own
        max_block_ms = 3000,                       # Max block time if buffer is full
        max_request_size = 256*1024,               # The maximum size of a request. This is also effectively a cap on the maximum record size. Note that the server has its own cap on record size which may be different from this
        max_in_flight_requests_per_connection = 1, # Requests are pipelined to kafka brokers up to this number of maximum requests per broker connection
        security_protocol = "PLAINTEXT"
    )

    # Setup accumulators and publish deadline
    accumulated_means = None
    accumulated_M2s = None
    accumulated_n_scans = 0
    frequencies = None
    producer_timestamps = []
    # producer_throughputs = []
    # producer_scans_per_batch = []
    batch_completion_times = []
    processing_times = []
    publish_deadline = time.monotonic() + publish_interval_sec

    producer_throughput = -1.0
    producer_scans = -1.0
    
    try:
        for msg in consumer:

            # Read metadata (msg header) provided by the producer
            if msg.headers:
                for key, value in msg.headers:
                    if key == "producer_ts":
                        producer_timestamp = float(value.decode('utf-8'))
                    elif key == "throughput":
                        producer_throughput = float(value.decode('utf-8'))
                    elif key == "scans_per_batch":
                        producer_scans = float(value.decode('utf-8'))

            producer_timestamps.append(producer_timestamp)
    
            # Process the batch
            processing_time_start = time.perf_counter()
            
            batch_result = process_batch(msg.value)
            batch_n_scans = batch_result["n_scans"]
            batch_means = batch_result["means"]
            batch_M2s = batch_result["M2s"]
    
            # Update accumulators
            if accumulated_n_scans == 0:
                frequencies= batch_result["frequencies"]
                accumulated_means = batch_means.copy()
                accumulated_M2s = batch_M2s.copy()
                accumulated_n_scans = batch_n_scans
            else:
                deltas = batch_means - accumulated_means
                total_n_scans = accumulated_n_scans + batch_n_scans
                accumulated_means += deltas * batch_n_scans / total_n_scans
                accumulated_M2s += batch_M2s + deltas**2 * accumulated_n_scans * batch_n_scans / total_n_scans
                accumulated_n_scans = total_n_scans
            
            # Batch processing finished
            processing_time = time.perf_counter() - processing_time_start
            processing_times.append(processing_time)
    
            # Publish results if publish_deadline surpassed
            now = time.monotonic()
            batch_completion_times.append(now)
            if now > publish_deadline:
                print("publishing")
                # Publish result
                result = {
                    "worker_id": worker_id,
                    "batches_details": {
                        "producer_timestamps": producer_timestamps,                           # Timestamp reported by producer for each batch
                        "waiting_times": [now - finish_time for finish_time in batch_completion_times], # How long each batch has been held due to publishing interval logic
                        "processing_times" : processing_times,                                # Time to process each batch
                        "throughput": producer_throughput,
                        "scans_per_batch": producer_scans
                    },
                    "results": {
                        "n_averaged_scans": accumulated_n_scans,
                        "frequencies": frequencies.tolist(),
                        "power_means": accumulated_means.tolist(),
                        "power_M2s": accumulated_M2s.tolist()
                    } 
                }
                producer.send(output_topic, value=result)
                producer.flush()
    
                # Reset accumulators and publish deadline
                accumulated_means = None
                accumulated_M2s = None
                accumulated_n_scans = 0
                frequencies = None
                producer_timestamps = []
                batch_completion_times = []
                processing_times = []
                #producer_throughputs = []
                #producer_scans_per_batch = []
                publish_deadline = now + publish_interval_sec
            else:
                print("skip publishing")
    finally:
        consumer.close()
        producer.close()

# Temp no daks

In [ ]:
worker_function(
        worker_id = 1,
        bootstrap_servers = BOOTSTRAP_SERVER,
        input_topic = TOPIC_STREAM,
        output_topic = TOPIC_RESULTS,
        publish_interval_sec = WORKER_PUBLISH_INTERVAL_SEC
)

# Start workers

In [6]:
def report_status(future):
    if future.status == "error":
        print(f"Task {future.key} failed:")
        print(future.exception())
    else:
        print(f"Task {future.key} finished with status {future.status}")

for (i, worker_ip) in enumerate(WORKER_IPS):
    future = dask_client.submit(
        worker_function, # function to be executed
        # Parameters to function
        worker_id = i + 1,
        bootstrap_servers = BOOTSTRAP_SERVER,
        input_topic = TOPIC_STREAM,
        output_topic = TOPIC_RESULTS,
        publish_interval_sec = WORKER_PUBLISH_INTERVAL_SEC,
        # Dask parameters
        key = f"worker{i+1}", # Identifier for the task
        workers = worker_ip, # Restrict execution to the specific worker
        resources = {}, # no resources (eg GPUs) required
    )
    future.add_done_callback(report_status)


Task worker2 failed:
KafkaConnectionError: 111 ECONNREFUSED


# Close resources

In [26]:
dask_client.close()
dask_cluster.close()

Task worker1 finished with status cancelled
Task worker2 finished with status cancelled
